<img src = "../assets/bbai-banner.png" align="center" height = "25%">

# MLflow Model Validation #

In [ ]:
import mlflow
import os
import sys
import pandas as pd
from mlflow.models import infer_signature

module_paths = [
    os.path.abspath(os.path.join('..')),
]
for module_path in module_paths:
    if module_path not in sys.path:
        sys.path.append(module_path)

from pdm.rul_model import RULModel
from pdm.config import RULConfig

In [ ]:
sensor_cols = ["s" + str(i) for i in set(range(1, 22))] + [
        "setting1",
        "setting2",
        "setting3",
    ]

sensor_cols = [
        x
        for x in sensor_cols
        if x not in ["s1", "s5", "s6", "s10", "s16", "s18", "s19", "setting3"]
    ]

# Set the tracking server uri for logging
# Don't forget to start the logging server!
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [ ]:
def open_data(file_path):
    # Identify the Column Names for the data
    column_names = ["id","cycle","setting1","setting2", "setting3",
            "s1","s2","s3","s4","s5","s6","s7","s8","s9","s10","s11","s12","s13","s14","s15","s16","s17","s18","s19","s20","s21",]

    # Load the file into Pandas dataframe
    train_data = pd.read_csv(file_path, sep=" ", header=None)
    #train_data.drop(train_data.columns[[26, 27]], axis=1, inplace=True)
    train_data.columns = column_names
    return train_data

In [ ]:
filename = "FD001"
filepath = '../data/training/train_'+filename+'.txt'
training_data = open_data(filepath)

print(training_data.shape)

filename = "FD001"
filepath = '../data/training/train_'+filename+'.txt'
training_data = open_data(filepath)

print(training_data.shape)

In [ ]:
   
filename = "FD001"

save_file = "C:/models/"+filename+".h5"

# Specify the file path
filepath = '../data/training/train_'+filename+'.txt'
print(filepath)


# Create a new MLflow Experiment
mlflow.set_experiment("RUL Prediction Tuning - CMAPSS "+filename)

mlflow.autolog(log_models=True, exclusive=True)
mlflow.tensorflow.autolog(log_models=True)

# # Baseline Model
# save_file = "C:/models/"+filename+"_baseline_reduced_features.h5"
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Baseline, Removed irrelevant features")
#     model.batch_train(training_data)
# mlflow.end_run()

# Baseline Model
save_file = "C:/models/"+filename+"_max_rul_reduced_features.h5"
cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, max_RUL=200)
model = RULModel(**cfg.model_dump())

training_data = open_data(filepath)

with mlflow.start_run() as run:
    
    mlflow.log_params(cfg.model_dump())
    
    mlflow.set_tag("Notes", "Max_RUL = 200, Removed irrelevant features")
    model.batch_train(training_data)
mlflow.end_run()

# Sequence Length = 90
save_file = "C:/models/"+filename+"_seq90_reduced_features.h5"
cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, seq_length=90)
model = RULModel(**cfg.model_dump())

training_data = open_data(filepath)

with mlflow.start_run() as run:
    
    mlflow.log_params(cfg.model_dump())
    
    mlflow.set_tag("Notes", "Sequence Length = 90, Removed irrelevant features")
    model.batch_train(training_data)
mlflow.end_run()

# # Sequence Length = 20, training patience = 10, attention_width=20, kernel size = 2
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, seq_length=20, kernel_size=2, training_patience=10, attention_width=20)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Sequence Length = 20")
#     model.batch_train(training_data)
# mlflow.end_run()


# # Sequence Length = 20
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, seq_length=20)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Sequence Length = 20")
#     model.batch_train(training_data)
# mlflow.end_run()

# # Sequence Length = 50
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, seq_length=50)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Sequence Length = 50")
#     model.batch_train(training_data)
# mlflow.end_run()

# # Kernel Size =2, Training Patience = 10, Attention width = 20.  Best of the best
# save_file = "C:/models/"+filename+"_best_reduced_features.h5"
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, kernel_size=2, training_patience=10, attention_width=20)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Removed irrelevant features, Kernel Size =2, Training Patience =10, Attention Width = 20")
#     model.batch_train(training_data)
# mlflow.end_run()

# # Kernel Size =2, Training Patience = 10, Attention width = 20, Max_RUL = 200.  Best of the best 2
# save_file = "C:/models/"+filename+"_best2_reduced_features.h5"
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, kernel_size=2, training_patience=10, attention_width=20, max_RUL=200)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Removed irrelevant features, Max_RUL = 200, Kernel Size =2, Training Patience =10, Attention Width = 20")
#     model.batch_train(training_data)
# mlflow.end_run()

# RULED OUT
# # LSTM Units = 32
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, lstm_units=32)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "LSTM Units = 32")
#     model.batch_train(training_data)
# mlflow.end_run()

# Kernel Size = 2
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, kernel_size=2)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Kernel Size = 2")
#     model.batch_train(training_data)
# mlflow.end_run()

# # Patience & Attention +
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, training_patience=10, attention_width=20)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Patience = 10 & Attention Width = 20")
#     model.batch_train(training_data)
# mlflow.end_run()


# # Increased Training Patience
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, training_patience=10)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Increased Patience (10)")
#     model.batch_train(training_data)
# mlflow.end_run()


# # Smaller Attention Width
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, attention_width=10)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Shorter Attention (10) ")
#     model.batch_train(training_data)
# mlflow.end_run()

# # Larger Attention Width
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, attention_width=20)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Longer Attention (20)")
#     model.batch_train(training_data)
# mlflow.end_run()

# RULED OUT
# Larger Optimizer Learning Rate
# cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, training_epochs=50, optimizer_learning_rate=.001)
# model = RULModel(**cfg.model_dump())

# training_data = open_data(filepath)

# with mlflow.start_run() as run:
    
#     mlflow.log_params(cfg.model_dump())
    
#     mlflow.set_tag("Notes", "Larger Optimizer Learning Rate (.001)")
#     model.batch_train(training_data)
# mlflow.end_run()